# Import necessary libraries

In [4]:
# Import libraries 
import pandas as pd
import requests
from scipy.optimize import minimize
import numpy as np

In [7]:
!pip install pulp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 11.0 MB/s  0:00:01 eta 0:00:01


In [8]:
import pulp

# Import, clean and prepare data

For the proyect, I took the data from transfermarkt and the official website of the Liga MX. However, the variables are different and there's no score. So I made the scoring function based on the premier league function. 

Let's begin by importing the raw data.

In [ ]:
raw_data = pd.read_excel('.../datos_completos.xlsx')
raw_data.head()

,Jugador,País,Posc,Equipo,Edad,Nacimiento,PJ_tiempo_jugado,Titular_tiempo_jugado,Mín_tiempo_jugado,90 s_tiempo_jugado,Gls._rendimiento,Ass_rendimiento,G+A_rendimiento,G-TP_rendimiento,TP_rendimiento,TPint_rendimiento,TA_rendimiento,TR_rendimiento,Gls._por_90_mins,Ast_por_90_mins,G+A_por_90_mins,G-TP_por_90_mins,G+A-TP_por_90_mins,Valor de mercado,Posc_PO,Posc_DF,Posc_CC,Posc_DL
0,Carlos Acevedo,MEX,PO,Santos,28-322,1996,26,26,"2,34",26.0,0,0,0,0,0,0,1,0,0.00,0.00,0.00,0.00,0.00,3800000,1,0,0,0
1,Rodrigo Aguirre,URU,"DL,CC",América,30-157,1994,15,6,659,7.3,5,1,6,5,0,0,6,0,0.68,0.14,0.82,0.68,0.82,3000000,0,0,1,1
2,Roberto Alvarado,MEX,"CC,DL",Guadalajara,26-181,1998,23,22,1776,19.7,8,6,14,5,3,4,3,0,0.41,0.30,0.71,0.25,0.56,7500000,0,0,1,1
3,Kevin Álvarez,MEX,DF,América,26-051,1999,14,10,973,10.8,0,1,1,0,0,0,1,1,0.00,0.09,0.09,0.00,0.09,5500000,0,1,0,0
4,Fidel Ambríz,MEX,CC,Monterrey,21-351,2003,18,10,991,11.0,1,0,1,1,0,0,2,0,0.09,0.00,0.09,0.09,0.09,6500000,0,0,1,0


Some players have two positions. We'll stay with the first one for this project. 

In [43]:
raw_data['Posc'] = raw_data['Posc'].str.split(',').str[0].str.strip()

In [44]:
print(raw_data.columns)

Index(['Jugador', 'País', 'Posc', 'Equipo', 'Edad', 'Nacimiento',
       'PJ_tiempo_jugado', 'Titular_tiempo_jugado', 'Mín_tiempo_jugado',
       '90 s_tiempo_jugado', 'Gls._rendimiento', 'Ass_rendimiento',
       'G+A_rendimiento', 'G-TP_rendimiento', 'TP_rendimiento',
       'TPint_rendimiento', 'TA_rendimiento', 'TR_rendimiento',
       'Gls._por_90_mins', 'Ast_por_90_mins', 'G+A_por_90_mins',
       'G-TP_por_90_mins', 'G+A-TP_por_90_mins', 'Valor de mercado', 'Posc_PO',
       'Posc_DF', 'Posc_CC', 'Posc_DL'],
      dtype='str')


Let's now map the position values to a number. Goalkeeper (Poertero "PO") is 1, Defence ("DF") is 2, Midfield (Centrocampista "CC") is 3 and forward (Delantero "DL") is 4.

In [ ]:
unique_values = raw_data["Posc"].unique()
print(unique_values)

['PO' 'DL' 'CC' 'DF']


In [ ]:
# Map the position values to numerical values
raw_data['position'] = raw_data['Posc'].map({'PO': 1, 'DF': 2, 'CC': 3, 'DL': 4})

Now we corect the market value to millions and decimals of millions. 

In [47]:
raw_data["Valor de mercado_en milloes"] = (pd.to_numeric(raw_data["Valor de mercado"])/1_000_000).round(2)
raw_data.head()

,Jugador,País,Posc,Equipo,Edad,Nacimiento,PJ_tiempo_jugado,Titular_tiempo_jugado,Mín_tiempo_jugado,90 s_tiempo_jugado,Gls._rendimiento,Ass_rendimiento,G+A_rendimiento,G-TP_rendimiento,TP_rendimiento,TPint_rendimiento,TA_rendimiento,TR_rendimiento,Gls._por_90_mins,Ast_por_90_mins,G+A_por_90_mins,G-TP_por_90_mins,G+A-TP_por_90_mins,Valor de mercado,Posc_PO,Posc_DF,Posc_CC,Posc_DL,position,Valor de mercado_en milloes
0,Carlos Acevedo,MEX,PO,Santos,28-322,1996,26,26,"2,34",26.0,0,0,0,0,0,0,1,0,0.00,0.00,0.00,0.00,0.00,3800000,1,0,0,0,1,3.8
1,Rodrigo Aguirre,URU,DL,América,30-157,1994,15,6,659,7.3,5,1,6,5,0,0,6,0,0.68,0.14,0.82,0.68,0.82,3000000,0,0,1,1,4,3.0
2,Roberto Alvarado,MEX,CC,Guadalajara,26-181,1998,23,22,1776,19.7,8,6,14,5,3,4,3,0,0.41,0.30,0.71,0.25,0.56,7500000,0,0,1,1,3,7.5
3,Kevin Álvarez,MEX,DF,América,26-051,1999,14,10,973,10.8,0,1,1,0,0,0,1,1,0.00,0.09,0.09,0.00,0.09,5500000,0,1,0,0,2,5.5
4,Fidel Ambríz,MEX,CC,Monterrey,21-351,2003,18,10,991,11.0,1,0,1,1,0,0,2,0,0.09,0.00,0.09,0.09,0.09,6500000,0,0,1,0,3,6.5


## ICT score and points

In Fantasy, an ICT (Influence, Creativity, Threat) index is used. Obviously, this doesn't exist in Liga MX. However, we can create a synthetic index using the data we have. This allows us to build different indices and compare them. 

In [48]:
raw_data["Influence"] = (
      5 * pd.to_numeric(raw_data["Gls._rendimiento"], errors="coerce")
    + 3 * pd.to_numeric(raw_data["Ass_rendimiento"], errors="coerce")
    + 0.01 * pd.to_numeric(raw_data["Mín_tiempo_jugado"], errors="coerce")
)

raw_data["Creativity"] = (
      5 * pd.to_numeric(raw_data["Ass_rendimiento"], errors="coerce")
    + 2 * pd.to_numeric(raw_data["Ast_por_90_mins"], errors="coerce")
)

raw_data["Threat"] = (
      5 * pd.to_numeric(raw_data["Gls._por_90_mins"], errors="coerce")
    + 2 * pd.to_numeric(raw_data["G+A_por_90_mins"], errors="coerce")
)

In [49]:
raw_data["ICT"] = (
    raw_data["Influence"]
    + raw_data["Creativity"]
    + raw_data["Threat"]
)

In [ ]:
# Normalize the ICT values using Min-Max scaling
from sklearn.preprocessing import MinMaxScaler 
raw_data["ICT_norm"] = MinMaxScaler().fit_transform(raw_data[["ICT"]])

In [ ]:
# For the points calculation, we will use the following formula:
raw_data["points"] = (
      6*pd.to_numeric(raw_data["Gls._rendimiento"], errors="coerce")
    + 4*pd.to_numeric(raw_data["Ass_rendimiento"], errors="coerce")
    + 0.02*pd.to_numeric(raw_data["Mín_tiempo_jugado"], errors="coerce")
    - 1*pd.to_numeric(raw_data["TA_rendimiento"], errors="coerce")
    - 3*pd.to_numeric(raw_data["TR_rendimiento"], errors="coerce")
)
raw_data['points_norm'] = MinMaxScaler().fit_transform(raw_data[["points"]])

Some players have missing values, so since the sample size is small, we'll fill them in with the mean.

In [52]:
raw_data["ICT_norm"] = raw_data["ICT_norm"].fillna(raw_data["ICT_norm"].mean())
raw_data["points_norm"] = raw_data["points_norm"].fillna(raw_data["points_norm"].mean())

In [53]:
raw_data.head()

,Jugador,País,Posc,Equipo,Edad,Nacimiento,PJ_tiempo_jugado,Titular_tiempo_jugado,Mín_tiempo_jugado,90 s_tiempo_jugado,Gls._rendimiento,Ass_rendimiento,G+A_rendimiento,G-TP_rendimiento,TP_rendimiento,TPint_rendimiento,TA_rendimiento,TR_rendimiento,Gls._por_90_mins,Ast_por_90_mins,G+A_por_90_mins,G-TP_por_90_mins,G+A-TP_por_90_mins,Valor de mercado,Posc_PO,Posc_DF,Posc_CC,Posc_DL,position,Valor de mercado_en milloes,Influence,Creativity,Threat,ICT,ICT_norm,points,points_norm
0,Carlos Acevedo,MEX,PO,Santos,28-322,1996,26,26,"2,34",26.0,0,0,0,0,0,0,1,0,0.00,0.00,0.00,0.00,0.00,3800000,1,0,0,0,1,3.8,NaN,0.00,0.00,NaN,0.233223,NaN,0.245113
1,Rodrigo Aguirre,URU,DL,América,30-157,1994,15,6,659,7.3,5,1,6,5,0,0,6,0,0.68,0.14,0.82,0.68,0.82,3000000,0,0,1,1,4,3.0,34.59,5.28,5.04,44.91,0.259907,41.18,0.232550
2,Roberto Alvarado,MEX,CC,Guadalajara,26-181,1998,23,22,1776,19.7,8,6,14,5,3,4,3,0,0.41,0.30,0.71,0.25,0.56,7500000,0,0,1,1,3,7.5,75.76,30.60,3.47,109.83,0.643300,104.52,0.597328
3,Kevin Álvarez,MEX,DF,América,26-051,1999,14,10,973,10.8,0,1,1,0,0,0,1,1,0.00,0.09,0.09,0.00,0.09,5500000,0,1,0,0,2,5.5,12.73,5.18,0.18,18.09,0.101518,19.46,0.107464
4,Fidel Ambríz,MEX,CC,Monterrey,21-351,2003,18,10,991,11.0,1,0,1,1,0,0,2,0,0.09,0.00,0.09,0.09,0.09,6500000,0,0,1,0,3,6.5,14.91,0.00,0.63,15.54,0.086458,23.82,0.132573


# PULP Implementqation

We construct vectors containing players' data including names, expected fantasy points ($p_i$), prices ($c_i$), positional roles, and club affiliations to select an optimal team.

Mathematically, we would state the problem as: 

Let $P$ be the set of all candidate players, where $x_i \in \{0, 1\}$ represents the decision variable indicating whether player $i$ is selected ($x_i = 1$) or not ($x_i = 0$).

$$\begin{aligned}
\text{maximize} \quad & \sum_{i \in P} p_i x_i \\
\text{subject to} \quad & \sum_{i \in P} c_i x_i \le 100.0 && \text{(Budget Constraint)} \\[8pt]
& \sum_{i \in \text{GK}} x_i = 2 && \text{(Goalkeepers)} \\[4pt]
& \sum_{i \in \text{DEF}} x_i = 5 && \text{(Defenders)} \\[4pt]
& \sum_{i \in \text{MID}} x_i = 5 && \text{(Midfielders)} \\[4pt]
& \sum_{i \in \text{FWD}} x_i = 3 && \text{(Forwards)} \\[8pt]
& \sum_{i \in T_k} x_i \le 3 \quad \forall k \in K && \text{(Team Limits)} \\[8pt]
\end{aligned}$$

In [54]:
players_name = list(raw_data['Jugador'])
points=list(raw_data['points_norm'])
price=list(raw_data['Valor de mercado_en milloes'])
positions=list(raw_data['position'])
team=list(raw_data['Equipo'])
num_players=len(raw_data)

In [56]:
# Define fpl budget to spend
total_budget=100

In [66]:
!pip install highspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 10.4 MB/s  0:00:00 eta 0:00:01


In [ ]:
# 1. Create the optimization problem
prob = pulp.LpProblem("Fantasy_Soccer_Optimizer", pulp.LpMaximize)

# 2. Create the decision variables (0 if not selected, 1 if selected)
player_vars = [
    pulp.LpVariable(f"player_{i}", cat="Binary") for i in range(num_players)
]

# 3. Objective Function: Maximize the normalized points
prob += (
    pulp.lpSum([points[i] * player_vars[i] for i in range(num_players)]),
    "Total_Points",
)

# 4. Restrictions

# Restriction A: Maximum Budget (Max Budget = 100.0)
prob += (
    pulp.lpSum([price[i] * player_vars[i] for i in range(num_players)]) <= 100.0,
    "Max_Budget",
)

# Restriction B: Exact Number of Players per Position
# 1 GK (PO), 5 DEF (DF), 5 MID (CC), 3 FWD (DL)
prob += (
    pulp.lpSum([
        player_vars[i] for i in range(num_players) if positions[i] == 1
    ])
    == 1,
    "Exactly_1_GK",
)
prob += (
    pulp.lpSum([
        player_vars[i] for i in range(num_players) if positions[i] == 2
    ])
    == 5,
    "Exactly_5_DEF",
)
prob += (
    pulp.lpSum([
        player_vars[i] for i in range(num_players) if positions[i] == 3
    ])
    == 5,
    "Exactly_5_MID",
)
prob += (
    pulp.lpSum([
        player_vars[i] for i in range(num_players) if positions[i] == 4
    ])
    == 3,
    "Exactly_3_FWD",
)

# Restriction C: Maximum 3 players per team
# Do not select more than 3 players from the same club in your lineup
unique_teams = set(team)
for t in unique_teams:
    prob += (
        pulp.lpSum([
            player_vars[i] for i in range(num_players) if team[i] == t
        ])
        <= 3,
        f"Max_3_players_{t}",
    )

# 5. Solve the optimization problem using the solver
'''
Here, you'll likely need to specify the path to the CBC solver if it isn't in your PATH. Make sure you have the CBC solver installed and know its location.

You might need to install Homebrew first and then install the CBC solver using brew install cbc.

When installing Homebrew, don't forget to set up the path correctly at the end by running:

echo >> /Users/_nombre user_/.zprofile
echo 'eval "$(/opt/homebrew/bin/brew shellenv zsh)"' >> /Users/_nombre user_/.zprofile
eval "$(/opt/homebrew/bin/brew shellenv zsh)" 

For example, if you're using Anaconda, the path might look something like "/path/to/anaconda3/bin/cbc". 
Adjust the path according to your installation.
'''

solver = pulp.COIN_CMD(path="/opt/homebrew/bin/cbc", msg=False)
prob.solve(solver)

# 6. Extract the selected players and their details
selected_indices = [
    i for i in range(num_players) if player_vars[i].varValue == 1.0
]
optimal_lineup = raw_data.iloc[selected_indices].copy()

print(f"State of the solution: {pulp.LpStatus[prob.status]}")
print(f"Total Average Points: {pulp.value(prob.objective):.2f}")
print(
    f"Total Cost: {sum(price[i] for i in selected_indices):.2f}M (of 100.0M)"
)
print(f"Selected Players: {len(selected_indices)} (of 14 required)")

# Show the optimal team
optimal_lineup[
    ['Jugador', 'position', 'Equipo', 'Valor de mercado_en milloes', 'points_norm']
]

Estado de la solución: Optimal
Puntos Totales Promedio: 7.30
Costo Total: 81.80M (de 100.0M)
Jugadores Seleccionados: 14 (de 14 requeridos)


,Jugador,position,Equipo,Valor de mercado_en milloes,points_norm
2,Roberto Alvarado,3,Guadalajara,7.5,0.597328
13,Germán Berterame,4,Monterrey,7.0,0.568648
19,Diber Cambindo,4,Necaxa,4.0,0.683253
21,Sergio Canales,3,Monterrey,7.0,0.776088
33,Willer Ditta,2,Cruz Azul,6.0,0.245113
41,Jesús Gallardo,2,Toluca,3.8,0.411311
53,Joaquim,2,UANL,5.0,0.313177
61,Kevin Mier,1,Cruz Azul,8.0,0.245113
62,Alan Mozo,2,Guadalajara,5.5,0.245113
65,Agustín Oliveros,2,Necaxa,3.5,0.274476


In [77]:
equipo_ideal = optimal_lineup.sort_values('position').reset_index(drop=True)
equipo_ideal[['Jugador', 'position', 'Equipo', 'Valor de mercado_en milloes']]

,Jugador,position,Equipo,Valor de mercado_en milloes
0,Kevin Mier,1,Cruz Azul,8.0
1,Willer Ditta,2,Cruz Azul,6.0
2,Jesús Gallardo,2,Toluca,3.8
3,Joaquim,2,UANL,5.0
4,Alan Mozo,2,Guadalajara,5.5
5,Agustín Oliveros,2,Necaxa,3.5
6,Roberto Alvarado,3,Guadalajara,7.5
7,Sergio Canales,3,Monterrey,7.0
8,José Paradela,3,Necaxa,5.0
9,Carlos Rotondi,3,Cruz Azul,6.0
